# Data Wrangling

In this notebook, I continued on the analysis of the data collected on the game: Arc Raiders through YouTube comments.

In [36]:
import numpy as np
import pandas as pd
import re
from datetime import datetime, timedelta
from scipy import stats
import os

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the dataset
data = pd.read_csv('../data/comments_data.csv')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 258443 entries, 0 to 258442
Data columns (total 30 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   comment_id         258443 non-null  str    
 1   text               258348 non-null  str    
 2   comment_date       258443 non-null  str    
 3   author_hash        258443 non-null  str    
 4   parent_id          60200 non-null   str    
 5   last_updated_at    258443 non-null  str    
 6   video_id           258443 non-null  str    
 7   likes              258443 non-null  int64  
 8   video_title        258443 non-null  str    
 9   video_date         258443 non-null  str    
 10  channel_id         258443 non-null  str    
 11  keyword_matched    258443 non-null  str    
 12  video_description  242347 non-null  str    
 13  char_count         258443 non-null  int64  
 14  word_count         258443 non-null  int64  
 15  avg_word_length    258443 non-null  float64
 16  has_url      

## 1. Data Cleaning

### 1.1. Handling Missing Values

In [38]:
data.isna().sum()

comment_id                0
text                     95
comment_date              0
author_hash               0
parent_id            198243
last_updated_at           0
video_id                  0
likes                     0
video_title               0
video_date                0
channel_id                0
keyword_matched           0
video_description     16096
char_count                0
word_count                0
avg_word_length           0
has_url                   0
has_mention               0
has_hashtag               0
exclamation_count         0
question_count            0
emoji_count               0
newline_count             0
uppercase_ratio           0
language                  0
day_of_week               0
day_num                   0
date                      0
is_spike                  0
z_score                2064
dtype: int64

In [39]:
data[data['video_description'].notna()]

,comment_id,text,comment_date,author_hash,parent_id,last_updated_at,video_id,likes,video_title,video_date,...,question_count,emoji_count,newline_count,uppercase_ratio,language,day_of_week,day_num,date,is_spike,z_score
0,UgxEgHTlwp52_t7tvld4AaABAg,this video is literally a lie - Arc Raiders do...,2026-03-12 09:00:00,00c95ebeeeb9b2094e02791c81e2ac9cd9e3a68724e2c5...,NaN,2026-03-12 09:00:00.000000,xuftkDxjGT4,1,Arc Raiders Reveal Trailer | Game Awards 2021,2021-12-10 03:58:14,...,0,0,0,0.080000,en,Thursday,3,2026-03-12,False,0.942388
1,UgxfQBIpu5Q9a2pqKD54AaABAg,Caraca não sabia que era tão antigo,2026-03-12 03:34:19,bef66a381d8737e126860408d372d8d4e8e463419de30a...,NaN,2026-03-12 03:34:19.000000,xuftkDxjGT4,0,Arc Raiders Reveal Trailer | Game Awards 2021,2021-12-10 03:58:14,...,0,0,0,0.028571,pt,Thursday,3,2026-03-12,False,0.942388
2,Ugx2KM9kJqjh_mesOIZ4AaABAg,😮This trailer is the greatest prequel lore in ...,2026-03-10 23:32:15,31c593066ff6110d558cc310a246c895d7bcf840d91e93...,NaN,2026-03-10 23:32:15.000000,xuftkDxjGT4,0,Arc Raiders Reveal Trailer | Game Awards 2021,2021-12-10 03:58:14,...,0,1,0,0.015152,en,Tuesday,1,2026-03-10,False,1.431188
3,Ugyt0PrH_pOeXqKrYRN4AaABAg,holy false advertising,2026-03-10 17:56:56,8890a908eaecc345c51e5e6716db3766e3b4044fbd0434...,NaN,2026-03-10 17:56:56.000000,xuftkDxjGT4,2,Arc Raiders Reveal Trailer | Game Awards 2021,2021-12-10 03:58:14,...,0,0,0,0.000000,no,Tuesday,1,2026-03-10,False,1.431188
4,UgxIu-puc-eEI2TVJ4h4AaABAg,Best game,2026-03-10 09:58:20,4ad74d9af134db04afdc8d76a0b60d2193b82d12ac29a3...,NaN,2026-03-10 09:58:20.000000,xuftkDxjGT4,0,Arc Raiders Reveal Trailer | Game Awards 2021,2021-12-10 03:58:14,...,0,0,0,0.111111,de,Tuesday,1,2026-03-10,False,1.431188
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258438,UgwwjVssxYAw2ATOKqB4AaABAg,Love the game. I'm getting burnt out as well. ...,2026-03-14 17:42:26,4058fafcd846b1cfa22f564d7ef567b71833c64f82cf54...,NaN,2026-03-14 17:42:26.000000,n1PbKZui6fU,0,Is Arc Raiders Dying?,2026-03-13 16:15:05,...,0,0,0,0.026549,en,Saturday,5,2026-03-14,False,-1.368747
258439,UgyyJQCzCSo_p5z44Ll4AaABAg,Feel the same way bro you would think they wou...,2026-03-13 19:42:30,da2da8153c00890cc2340cb57bdfed0e776d96b074611d...,NaN,2026-03-13 19:42:30.000000,n1PbKZui6fU,0,Is Arc Raiders Dying?,2026-03-13 16:15:05,...,0,0,0,0.019802,en,Friday,4,2026-03-13,False,0.489446
258440,UgyPvIRpPl0gVop3wg94AaABAg,You streamers that play all damn day surely ar...,2026-03-13 16:26:50,3476d75005117e92b86e9e92431bf3a5b75aeec1ad8177...,NaN,2026-03-13 16:26:50.000000,n1PbKZui6fU,2,Is Arc Raiders Dying?,2026-03-13 16:15:05,...,0,0,0,0.015385,en,Friday,4,2026-03-13,False,0.489446
258441,UgxOkPb7SaoGNA6yAyh4AaABAg,Nah bro 600 hrs no wonder youre so burnt out,2026-03-13 16:24:32,191067ee8ed485801dcddbfa19ad217135526d74731023...,NaN,2026-03-13 16:24:32.000000,n1PbKZui6fU,3,Is Arc Raiders Dying?,2026-03-13 16:15:05,...,0,0,0,0.022727,en,Friday,4,2026-03-13,False,0.489446


In [40]:
# Filter out rows with missing comments
filtered_data = data[data['text'].notna()]
filtered_data.info()

<class 'pandas.DataFrame'>
Index: 258348 entries, 0 to 258442
Data columns (total 30 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   comment_id         258348 non-null  str    
 1   text               258348 non-null  str    
 2   comment_date       258348 non-null  str    
 3   author_hash        258348 non-null  str    
 4   parent_id          60179 non-null   str    
 5   last_updated_at    258348 non-null  str    
 6   video_id           258348 non-null  str    
 7   likes              258348 non-null  int64  
 8   video_title        258348 non-null  str    
 9   video_date         258348 non-null  str    
 10  channel_id         258348 non-null  str    
 11  keyword_matched    258348 non-null  str    
 12  video_description  242259 non-null  str    
 13  char_count         258348 non-null  int64  
 14  word_count         258348 non-null  int64  
 15  avg_word_length    258348 non-null  float64
 16  has_url           

In [41]:
# Even if not null, some video descriptions, text or video_title might be empty strings. Let's check for that.
#filtered_data['video_description'].apply(lambda x: len(x.strip()) == 0).sum()
#filtered_data['text'].apply(lambda x: len(x.strip()) == 0).sum()
#filtered_data['video_title'].apply(lambda x: len(x.strip()) == 0).sum()
filtered_data = filtered_data[~(filtered_data['text'].apply(lambda x: len(x.strip()) == 0) | 
                                filtered_data['video_title'].apply(lambda x: len(x.strip()) == 0))]
filtered_data.shape

(258348, 30)

In [42]:
data = filtered_data.copy()

### 1.2. Standardizing Datetime

In [43]:
data['comment_date'] = pd.to_datetime(data['comment_date'])
data['video_date'] = pd.to_datetime(data['video_date'])
data['last_updated_at'] = pd.to_datetime(data['last_updated_at'])

In [44]:
# Sort by comment date for time series analysis
data = data.sort_values('comment_date').reset_index(drop=True)

print("Date columns converted:")
print(f"  Comment date range: {data['comment_date'].min()} to {data['comment_date'].max()}")
print(f"  Video date range: {data['video_date'].min()} to {data['video_date'].max()}")
print(f"  Total time span: {(data['comment_date'].max() - data['comment_date'].min()).days} days")

Date columns converted:
  Comment date range: 2021-12-10 03:58:59 to 2026-03-14 21:06:08
  Video date range: 2021-12-10 03:58:14 to 2026-03-13 18:50:37
  Total time span: 1555 days


## 2. Defining Features

### 2.1. Event Mention Detection

Based on Arc Raiders timeline and major announcements:
- **GenAI Voice Announcement**: When Embark Studios announced AI-generated voices
- **Business Model Change**: When they announced changing from free-to-play to paid
- **Game Launch**: Official release date

These dates will be used to calculate temporal features and decay functions.

In [45]:
MAJOR_EVENTS = {
    'game_announcement': pd.Timestamp('2021-12-10'),
    'genai_announcement': pd.Timestamp('2024-05-15'),
    'business_model_change': pd.Timestamp('2024-08-20'),
    'game_launch': pd.Timestamp('2025-10-30'),
    'early_access': pd.Timestamp('2025-11-15'),
}
print("Major Event Timeline:")
print("=" * 60)
for event, date in MAJOR_EVENTS.items():
    print(f"  {event:25s}: {date.strftime('%Y-%m-%d')}")
print("=" * 60)

Major Event Timeline:
  game_announcement        : 2021-12-10
  genai_announcement       : 2024-05-15
  business_model_change    : 2024-08-20
  game_launch              : 2025-10-30
  early_access             : 2025-11-15


Create binary features to detect if comments mention specific events.
This helps us identify which comments are directly responding to announcements.

In [46]:
EVENT_KEYWORDS = {
    'game_announcement': [
        'announcement', 'announced', 'reveal', 'revealed', 'teaser', 'teased',
        'first look', 'sneak peek', 'game announcement', 'game reveal'
    ],
    'genai': [
        'ai voice', 'ai-generated', 'artificial intelligence', 'generated voice',
        'ai audio', 'synthetic voice', 'ai narration', 'ai dialogue',
        'voice ai', 'ai-voiced', 'ai acting', 'machine voice'
    ],
    'business_model': [
        'free to play', 'f2p', 'freemium', 'paid game', 'pay to play', 'p2p',
        'price', 'cost', 'buying', 'purchase', 'buy the game', 'not free',
        'charging', 'monetization', 'business model', 'payment model'
    ],
    'launch': [
        'launch', 'release', 'released', 'launching', 'came out', 'available',
        'live', 'went live', 'early access'
    ]
}

In [47]:
CORE_GAME_KEYWORDS = {
    'ai_usage': [
        'ai voice','ai voices','ai-generated','ai generated',
        'artificial intelligence','ai audio',
        'synthetic voice','ai narration',
        'ai dialogue','voice ai',
        'ai acting','ai voice acting',
        'machine voice','tts',
        'text to speech','deepfake voice',
        'ai characters','ai content',
        'procedural voice', 'elevenlabs',
        'ai writing','ai script',
        'ai npcs','ai npc',
        'ai animation',
        'generative ai','gen ai'
    ],
    'ai_reaction': [
        'hate ai','dont like ai',
        'ai is bad','ai is good',
        'lazy devs','cheap shortcut',
        'no soul','soulless',
        'ruins immersion',
        'fine with ai',
        'okay with ai',
        'dont care about ai',
        'ai is the future',
        'ai saves money',
        'good use of ai',
        'bad use of ai',
        'ai controversy'
    ],
    'gameplay': [
        'gameplay','mechanics',
        'gunplay','combat',
        'movement','controls',
        'difficulty',
        'fun gameplay',
        'boring gameplay',
        'playstyle',
        'strategy','tactics',
        'loop','game loop'
    ],
    'extraction_systems': [
        'extract','extraction',
        'exfil','raid',
        'loot','looting',
        'gear','loadout',
        'stash','inventory',
        'progression',
        'grind','grindy',
        'crafting','materials',
        'resources',
        'risk reward',
        'gear loss'
    ],
    'pvp_structure': [
        'pvp','pve','pvpve',
        'solo','squad',
        'team','teammates',
        'coop','co op',
        'multiplayer',
        'matchmaking',
        'player interaction',
        'kill on sight',
        'camping'
    ],
    'balance': [
        'balance','balanced',
        'unbalanced',
        'overpowered','op',
        'underpowered',
        'nerf','buff',
        'broken',
        'fair','unfair',
        'skill issue'
    ],
    'monetization': [
        'free to play','f2p',
        'price','cost',
        'worth it',
        'microtransactions',
        'battle pass',
        'season pass',
        'cosmetics',
        'skins',
        'pay to win','p2w',
        'dlc'
    ],
    'performance': [
        'fps','performance',
        'optimization',
        'lag','stutter',
        'crash',
        'server issues',
        'disconnect',
        'ping',
        'graphics',
        'settings'
    ],
    'updates': [
        'launch','release',
        'early access',
        'patch','update',
        'hotfix',
        'patch notes',
        'maintenance',
        'bug fix'
    ],
    'announcement': [
        'trailer',
        'announcement',
        'reveal',
        'gameplay trailer',
        'dev update',
        'roadmap',
        'showcase',
        'preview',
        'announced', 'revealed', 'teaser', 'teased',
        'first look', 'sneak peek', 'game announcement', 'game reveal',
        'launch', 'release', 'released', 'launching', 'came out', 'available',
        'live', 'went live', 'early access'
    ],
    'comparison': [
        'like tarkov',
        'like destiny',
        'like division',
        'like helldivers',
        'similar to',
        'copy of',
        'ripoff',
        'inspired by'
    ],
    'sentiment': [
        'looks good',
        'looks bad',
        'excited',
        'hyped',
        'disappointed',
        'worried',
        'concerned',
        'hope this succeeds',
        'dead game'
    ]
}

In [48]:
def detect_event_mention(text, keywords):
    """Detect if text mentions any of the keywords"""
    if pd.isna(text):
        return False
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in keywords)

In [49]:
# Create event mention columns
# Combine AI usage + AI reaction keywords into a single set to search for GenAI discussions.
genai_keywords = CORE_GAME_KEYWORDS['ai_usage'] + CORE_GAME_KEYWORDS['ai_reaction']
# Combine business model related keywords
business_keywords = CORE_GAME_KEYWORDS['pvp_structure'] + CORE_GAME_KEYWORDS['monetization'] + EVENT_KEYWORDS['business_model']
# Combine launch and updates related keywords
launch_related = CORE_GAME_KEYWORDS['updates'] + CORE_GAME_KEYWORDS['announcement'] + EVENT_KEYWORDS['launch']
# For any game related mentions
anything_game = CORE_GAME_KEYWORDS['ai_reaction'] + CORE_GAME_KEYWORDS['ai_usage'] + CORE_GAME_KEYWORDS['announcement'] + CORE_GAME_KEYWORDS['balance'] + CORE_GAME_KEYWORDS['comparison'] + CORE_GAME_KEYWORDS['extraction_systems'] + CORE_GAME_KEYWORDS['gameplay'] + CORE_GAME_KEYWORDS['monetization'] + CORE_GAME_KEYWORDS['performance'] + CORE_GAME_KEYWORDS['pvp_structure'] + CORE_GAME_KEYWORDS['sentiment'] + CORE_GAME_KEYWORDS['updates'] + EVENT_KEYWORDS['business_model'] + EVENT_KEYWORDS['game_announcement'] + EVENT_KEYWORDS['genai'] + EVENT_KEYWORDS['launch']

data['mentions_genai'] = data['text'].apply(
    lambda x: detect_event_mention(x, genai_keywords)
)
data['mentions_business_model'] = data['text'].apply(
    lambda x: detect_event_mention(x, business_keywords)
)

data['mentions_launch'] = data['text'].apply(
    lambda x: detect_event_mention(x, launch_related)
)

data['mentions_game'] = data['text'].apply(
    lambda x: detect_event_mention(x, anything_game)
)

In [70]:
# Summary of event mentions
print("Event Mention Detection Results:")
print("=" * 60)
print(f"Comments mentioning GenAI:           {data['mentions_genai'].sum():>8,} ({data['mentions_genai'].mean()*100:>5.2f}%)")
print(f"Comments mentioning Business Model:  {data['mentions_business_model'].sum():>8,} ({data['mentions_business_model'].mean()*100:>5.2f}%)")
print(f"Comments mentioning Launch:          {data['mentions_launch'].sum():>8,} ({data['mentions_launch'].mean()*100:>5.2f}%)")
print(f"Comments mentioning the game:        {data['mentions_game'].sum():>8,}")
print("=" * 60)

Event Mention Detection Results:
Comments mentioning GenAI:                557 ( 0.22%)
Comments mentioning Business Model:    25,707 ( 9.95%)
Comments mentioning Launch:             8,575 ( 3.32%)
Comments mentioning the game:          84,607


### 2.2. Temporal Distance from Events

We calculate how many days each comment was posted relative to each major event.
This is crucial for measuring sentiment decay over time.

In [51]:
# Calculate days from each event
for event_name, event_date in MAJOR_EVENTS.items():
    col_name = f'days_from_{event_name}'
    data[col_name] = (data['comment_date'] - event_date).dt.days

In [52]:
# Display temporal features
print("Temporal Distance Features Created:")
print("=" * 40)
for event_name in MAJOR_EVENTS.keys():
    col_name = f'days_from_{event_name}'
    if col_name in data.columns:
        print(f"\n{event_name}:")
        print(f"  Min: {data[col_name].min():>6.0f} days (before event)")
        print(f"  Max: {data[col_name].max():>6.0f} days (after event)")
        print(f"  Comments before event: {(data[col_name] < 0).sum():>8,}")
        print(f"  Comments after event:  {(data[col_name] >= 0).sum():>8,}")

Temporal Distance Features Created:

game_announcement:
  Min:      0 days (before event)
  Max:   1555 days (after event)
  Comments before event:        0
  Comments after event:   258,348

genai_announcement:
  Min:   -887 days (before event)
  Max:    668 days (after event)
  Comments before event:   11,582
  Comments after event:   246,766

business_model_change:
  Min:   -984 days (before event)
  Max:    571 days (after event)
  Comments before event:   12,211
  Comments after event:   246,137

game_launch:
  Min:  -1420 days (before event)
  Max:    135 days (after event)
  Comments before event:   22,861
  Comments after event:   235,487

early_access:
  Min:  -1436 days (before event)
  Max:    119 days (after event)
  Comments before event:   23,239
  Comments after event:   235,109


### 2.3 Engagement Related Features

Here we create engagement-related features using like_count and other metrics.
These will be used for popularity-weighted sentiment scores.

In [53]:
data['has_likes'] = data['likes'] > 0
data['like_count_log'] = np.log1p(data['likes'])  # Log transform for skewed distribution

In [54]:
data['engagement_tier'] = pd.cut(
    data['likes'],
    bins=[-1, 0, 5, 20, 100, float('inf')],
    labels=['no_likes', 'low_engagement', 'medium_engagement', 'high_engagement', 'viral'])

In [55]:
data['comment_latency_days'] = (data['comment_date'] - data['video_date']).dt.total_seconds() / (24 * 3600)
data['comment_latency_hours'] = (data['comment_date'] - data['video_date']).dt.total_seconds() / 3600

In [56]:
data['latency_category'] = pd.cut(
    data['comment_latency_days'],
    bins=[-float('inf'), 0, 1, 7, 30, float('inf')],
    labels=['before_upload', 'same_day', 'within_week', 'within_month', 'after_month'])

In [57]:
print("Engagement Features Summary:")
print("=" * 40)
print(f"Comments with likes:        {data['has_likes'].sum():>8,} ({data['has_likes'].mean()*100:>5.2f}%)")
print(f"Mean like count:            {data['likes'].mean():>12.2f}")
print(f"Median like count:          {data['likes'].median():>12.2f}")
print(f"Max like count:             {data['likes'].max():>12.0f}")
print(f"\nEngagement Tier Distribution:")
for tier in data['engagement_tier'].cat.categories:
    count = (data['engagement_tier'] == tier).sum()
    print(f"  {tier:20s}: {count:>8,} ({count/len(data)*100:>5.2f}%)")

Engagement Features Summary:
Comments with likes:          72,453 (28.04%)
Mean like count:                    6.61
Median like count:                  0.00
Max like count:                    22393

Engagement Tier Distribution:
  no_likes            :  185,895 (71.96%)
  low_engagement      :   56,676 (21.94%)
  medium_engagement   :    9,206 ( 3.56%)
  high_engagement     :    4,333 ( 1.68%)
  viral               :    2,238 ( 0.87%)


## 3. Text Preprocessing

Preparing the text to make it suitable for sentiment analysis with RoBERTa model.

In [58]:
def preprocess_for_roberta(text):
    """
    Preprocess text for RoBERTa sentiment analysis.
    - Replace URLs with [URL] token
    - Replace @mentions with [USER] token
    - Remove excessive whitespace
    """
    if pd.isna(text):
        return ""
    
    # Replace URLs
    text = re.sub(r'http\S+|www\S+', '[URL]', text)
    
    # Replace @mentions
    text = re.sub(r'@\w+', '[USER]', text)
    
    # Clean whitespace
    text = ' '.join(text.split())
    
    return text

In [59]:
data['text_preprocessed'] = data['text'].apply(preprocess_for_roberta)

In [60]:
# Check preprocessing
print("Text Preprocessing Complete:")
print("=" * 60)
print(f"Original text with URLs:     {data['has_url'].sum():>6,}")
print(f"Original text with mentions: {data['has_mention'].sum():>6,}")
print(f"\nSample preprocessed texts:")
print("-" * 60)
for i, row in data[data['has_url'] | data['has_mention']].head(3).iterrows():
    print(f"\nOriginal:     {row['text'][:100]}...")
    print(f"Preprocessed: {row['text_preprocessed'][:100]}...")

Text Preprocessing Complete:
Original text with URLs:        377
Original text with mentions: 16,685

Sample preprocessed texts:
------------------------------------------------------------

Original:     @seanrobinson7748 thanks brother...
Preprocessed: [USER] thanks brother...

Original:     @ras6794 playing as a feMaLe is immersion breaking and cringey. So why do you care what I think?...
Preprocessed: [USER] playing as a feMaLe is immersion breaking and cringey. So why do you care what I think?...

Original:     @player-cw9nj Thanks...
Preprocessed: [USER]-cw9nj Thanks...


## 4. Adding context

Adding context from the video_description, as well as the top-level comment for replies, gives a **Conversational NLP** edge to my project. Even though it is not relevant for my RoBERTa-based model, it works great for the LLMs I am going to work with.

In [61]:
data['video_context'] = data['video_description'].str[:500]

In [62]:
# Creating the Parent Text Mapping first
text_map = data.set_index('comment_id')['text'].to_dict()

# Mapping parent texts to replies
data['parent_text'] = data['parent_id'].map(text_map)

## 5. Filtering the data for analysis

### 5.1 Keeping only the high quality data

In [63]:
# High-quality English comments only
data_clean = data[
    (data['language'] == 'en') &
    (data['word_count'] >= 3) &
    (~data['text'].isna())
].copy()

In [64]:
# Bot Removal (Deduplicate identical text)
data_clean = data_clean.drop_duplicates(subset=['text'], keep='first')
data_clean.info()

<class 'pandas.DataFrame'>
Index: 189726 entries, 0 to 258347
Data columns (total 48 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   comment_id                       189726 non-null  str           
 1   text                             189726 non-null  str           
 2   comment_date                     189726 non-null  datetime64[us]
 3   author_hash                      189726 non-null  str           
 4   parent_id                        41698 non-null   str           
 5   last_updated_at                  189726 non-null  datetime64[us]
 6   video_id                         189726 non-null  str           
 7   likes                            189726 non-null  int64         
 8   video_title                      189726 non-null  str           
 9   video_date                       189726 non-null  datetime64[us]
 10  channel_id                       189726 non-null  str       

### 5.2 Trimming the unnecessary columns from the dataset

In [65]:
column_list = data_clean.columns.tolist()
print(column_list)

['comment_id', 'text', 'comment_date', 'author_hash', 'parent_id', 'last_updated_at', 'video_id', 'likes', 'video_title', 'video_date', 'channel_id', 'keyword_matched', 'video_description', 'char_count', 'word_count', 'avg_word_length', 'has_url', 'has_mention', 'has_hashtag', 'exclamation_count', 'question_count', 'emoji_count', 'newline_count', 'uppercase_ratio', 'language', 'day_of_week', 'day_num', 'date', 'is_spike', 'z_score', 'mentions_genai', 'mentions_business_model', 'mentions_launch', 'mentions_game', 'days_from_game_announcement', 'days_from_genai_announcement', 'days_from_business_model_change', 'days_from_game_launch', 'days_from_early_access', 'has_likes', 'like_count_log', 'engagement_tier', 'comment_latency_days', 'comment_latency_hours', 'latency_category', 'text_preprocessed', 'video_context', 'parent_text']


In [66]:
columns_to_keep = [
    'comment_id', 'parent_id', 'parent_text',  # Conversation context
    'text_preprocessed',                       # Model input
    'video_title', 'video_context', 'video_id',# Video context
    'comment_date', 'author_hash',             # Temporal/Identity
    'days_from_genai_announcement', 
    'days_from_business_model_change', 
    'days_from_game_announcement',
    'days_from_early_access',
    'days_from_game_launch',                   # Analysis variables
    'mentions_genai', 'mentions_business_model',
    'mentions_game', 
    'likes', 'like_count_log',                 # Engagement
    'is_spike', 'z_score']                     # Spike detection

In [67]:
data_final = data_clean[columns_to_keep].copy()

In [ ]:
DATA_DIR = os.path.join("..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

data_final.to_csv(os.path.join(DATA_DIR, 'comments_for_analysis.csv'), index=False)
print(f"Dataset prepared with {len(data_final)} rows.")

Dataset prepared with 189726 rows.
